In [4]:

import os
import glob
import csv
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# DAY 16: RAYLEIGH NUMBER COMPARISON
# --------------------------------------------------

# Find simulation folder
folders = glob.glob("combined_mantle_results*")

if not folders:
    raise FileNotFoundError(
        "Run the combined mantle convection model first."
    )

result_folder = max(folders, key=os.path.getmtime)
files = glob.glob(os.path.join(result_folder, "*.npz"))

if not files:
    raise FileNotFoundError(
        "No .npz simulation files found in " + result_folder
    )

output_folder = "day16_rayleigh_comparison"
os.makedirs(output_folder, exist_ok=True)

summary = []
datasets = []

# --------------------------------------------------
# 1. LOAD AND ANALYSE SIMULATION FILES
# --------------------------------------------------

for filename in files:

    with np.load(filename) as data:

        required = ["ra", "x", "z", "T", "u", "w"]

        if not all(key in data for key in required):
            print("Skipping incomplete file:", filename)
            continue

        Ra = float(np.asarray(data["ra"]).item())

        x = data["x"].copy()
        z = data["z"].copy()
        T = data["T"].copy()
        u = data["u"].copy()
        w = data["w"].copy()

    # Check dimensions
    if T.shape != u.shape or T.shape != w.shape:
        print("Skipping incompatible arrays:", filename)
        continue

    if T.shape != (len(z), len(x)):
        print("Skipping unexpected array shape:", filename)
        continue

    # Check finite values
    if not all(
        np.all(np.isfinite(arr))
        for arr in [x, z, T, u, w]
    ):
        print("Skipping non-finite data:", filename)
        continue

    if len(x) < 3 or len(z) < 3:
        print("Skipping grid that is too small:", filename)
        continue

    dx = float(np.mean(np.diff(x)))
    dz = float(np.mean(np.diff(z)))

    if dx <= 0 or dz <= 0:
        print("Skipping invalid grid spacing:", filename)
        continue

    # ----------------------------------------------
    # VELOCITY
    # ----------------------------------------------

    speed = np.sqrt(u**2 + w**2)

    max_speed = np.max(speed)
    mean_speed = np.mean(speed)

    interior_speed = speed[1:-1, 1:-1]

    rms_speed = np.sqrt(
        np.mean(interior_speed**2)
    )

    # ----------------------------------------------
    # NUSSELT NUMBER
    # ----------------------------------------------

    # Temperature gradient at bottom boundary
    bottom_gradient = (
        T[1, 1:-1] - T[0, 1:-1]
    ) / dz

    # Temperature gradient at top boundary
    top_gradient = (
        T[-1, 1:-1] - T[-2, 1:-1]
    ) / dz

    Nu_bottom = -np.mean(bottom_gradient)
    Nu_top = -np.mean(top_gradient)

    denominator = max(
        abs(Nu_bottom),
        abs(Nu_top),
        1e-12
    )

    mismatch = (
        abs(Nu_bottom - Nu_top) / denominator
    ) * 100

    # ----------------------------------------------
    # TEMPERATURE RANGE
    # ----------------------------------------------

    Tmin = np.min(T)
    Tmax = np.max(T)

    # ----------------------------------------------
    # SAVE DATA IN LISTS
    # ----------------------------------------------

    summary.append([
        Ra,
        max_speed,
        mean_speed,
        rms_speed,
        Nu_bottom,
        Nu_top,
        mismatch,
        Tmin,
        Tmax
    ])

    datasets.append({
        "Ra": Ra,
        "x": x,
        "z": z,
        "T": T,
        "u": u,
        "w": w,
        "speed": speed
    })

    print("\nRayleigh number:", Ra)
    print("Maximum speed:", max_speed)
    print("Mean speed:", mean_speed)
    print("RMS interior speed:", rms_speed)
    print("Bottom Nusselt number:", Nu_bottom)
    print("Top Nusselt number:", Nu_top)
    print("Heat-flux mismatch (%):", mismatch)
    print("Temperature range:", Tmin, "to", Tmax)

# Stop if no valid data
if not datasets:
    raise ValueError(
        "No valid simulation datasets were found."
    )

# Sort by Rayleigh number
datasets.sort(key=lambda item: item["Ra"])
summary.sort(key=lambda row: row[0])

Ra_values = [row[0] for row in summary]
max_speeds = [row[1] for row in summary]
mean_speeds = [row[2] for row in summary]
rms_speeds = [row[3] for row in summary]
Nu_bottom_values = [row[4] for row in summary]
Nu_top_values = [row[5] for row in summary]
mismatches = [row[6] for row in summary]

# --------------------------------------------------
# 2. TEMPERATURE FIELD COMPARISON
# --------------------------------------------------

fig, axes = plt.subplots(
    1,
    len(datasets),
    figsize=(5 * len(datasets), 4),
    squeeze=False
)

for i, item in enumerate(datasets):

    ax = axes[0, i]

    contour = ax.contourf(
        item["x"],
        item["z"],
        item["T"],
        levels=25,
        cmap="inferno"
    )

    ax.set_title(f"Ra = {item['Ra']:g}")
    ax.set_xlabel("Horizontal position, x")
    ax.set_ylabel("Vertical position, z")

fig.colorbar(
    contour,
    ax=axes.ravel().tolist(),
    label="Normalized temperature",
    pad=0.03
)

fig.suptitle("Day 16: Temperature comparison")

# Use ONE layout method only
fig.subplots_adjust(
    top=0.82,
    bottom=0.15,
    left=0.08,
    right=0.90,
    wspace=0.35
)

fig.savefig(
    os.path.join(
        output_folder,
        "day16_temperature_fields.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)

# --------------------------------------------------
# 3. VELOCITY MAGNITUDE COMPARISON
# --------------------------------------------------

fig, axes = plt.subplots(
    1,
    len(datasets),
    figsize=(5 * len(datasets), 4),
    squeeze=False
)

for i, item in enumerate(datasets):

    ax = axes[0, i]

    contour = ax.contourf(
        item["x"],
        item["z"],
        item["speed"],
        levels=25,
        cmap="viridis"
    )

    ax.set_title(f"Ra = {item['Ra']:g}")
    ax.set_xlabel("Horizontal position, x")
    ax.set_ylabel("Vertical position, z")

fig.colorbar(
    contour,
    ax=axes.ravel().tolist(),
    label="Velocity magnitude",
    pad=0.03
)

fig.suptitle("Day 16: Velocity comparison")

fig.subplots_adjust(
    top=0.82,
    bottom=0.15,
    left=0.08,
    right=0.90,
    wspace=0.35
)

fig.savefig(
    os.path.join(
        output_folder,
        "day16_velocity_fields.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close(fig)

# --------------------------------------------------
# 4. VELOCITY VERSUS RAYLEIGH NUMBER
# --------------------------------------------------

plt.figure(figsize=(7, 5))

plt.plot(
    Ra_values,
    max_speeds,
    "o-",
    label="Maximum speed"
)

plt.plot(
    Ra_values,
    mean_speeds,
    "s--",
    label="Mean speed"
)

plt.plot(
    Ra_values,
    rms_speeds,
    "^:",
    label="RMS interior speed"
)

plt.xlabel("Rayleigh number")
plt.ylabel("Velocity magnitude")
plt.title("Day 16: Velocity versus Rayleigh number")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        output_folder,
        "day16_velocity_vs_Ra.png"
    ),
    dpi=200
)

plt.close()

# --------------------------------------------------
# 5. NUSSELT NUMBER VERSUS RAYLEIGH NUMBER
# --------------------------------------------------

plt.figure(figsize=(7, 5))

plt.plot(
    Ra_values,
    Nu_bottom_values,
    "o-",
    label="Bottom Nu"
)

plt.plot(
    Ra_values,
    Nu_top_values,
    "s--",
    label="Top Nu"
)

plt.axhline(
    1.0,
    linestyle=":",
    label="Conductive reference (Nu = 1)"
)

plt.xlabel("Rayleigh number")
plt.ylabel("Nusselt number")
plt.title("Day 16: Heat transfer versus Rayleigh number")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        output_folder,
        "day16_Nu_vs_Ra.png"
    ),
    dpi=200
)

plt.close()

# --------------------------------------------------
# 6. HEAT-FLUX MISMATCH
# --------------------------------------------------

plt.figure(figsize=(7, 5))

plt.plot(
    Ra_values,
    mismatches,
    "o-"
)

plt.xlabel("Rayleigh number")
plt.ylabel("Top-bottom heat-flux mismatch (%)")
plt.title("Day 16: Heat-flux consistency")
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        output_folder,
        "day16_flux_mismatch.png"
    ),
    dpi=200
)

plt.close()

# --------------------------------------------------
# 7. SAVE SUMMARY CSV
# --------------------------------------------------

csv_path = os.path.join(
    output_folder,
    "day16_rayleigh_summary.csv"
)

headers = [
    "Rayleigh_number",
    "Maximum_speed",
    "Mean_speed",
    "RMS_interior_speed",
    "Nu_bottom",
    "Nu_top",
    "Heat_flux_mismatch_percent",
    "Minimum_temperature",
    "Maximum_temperature"
]

with open(csv_path, "w", newline="") as file:

    writer = csv.writer(file)
    writer.writerow(headers)
    writer.writerows(summary)

# --------------------------------------------------
# 8. COMPLETION MESSAGE
# --------------------------------------------------

print("\n" + "=" * 45)
print("DAY 16 ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 45)

print("Input folder:", result_folder)
print("Output folder:", output_folder)
print("Summary CSV:", csv_path)

print("\nGenerated files:")
print("1. day16_temperature_fields.png")
print("2. day16_velocity_fields.png")
print("3. day16_velocity_vs_Ra.png")
print("4. day16_Nu_vs_Ra.png")
print("5. day16_flux_mismatch.png")
print("6. day16_rayleigh_summary.csv")

print("\nAll available results have been processed.")



Rayleigh number: 1000.0
Maximum speed: 261.5621970066285
Mean speed: 83.09220657473846
RMS interior speed: 93.76076219471409
Bottom Nusselt number: 5.836153095734757
Top Nusselt number: 7.763581696574158
Heat-flux mismatch (%): 24.826538525252055
Temperature range: 0.0 to 1.0

Rayleigh number: 2500.0
Maximum speed: 564.3629566969514
Mean speed: 166.2647986416392
RMS interior speed: 183.25955899944853
Bottom Nusselt number: 9.746131680905368
Top Nusselt number: 12.454051976431867
Heat-flux mismatch (%): 21.743287250213715
Temperature range: 0.0 to 1.0

Rayleigh number: 500.0
Maximum speed: 133.09757056314587
Mean speed: 41.68883220891317
RMS interior speed: 46.9836503556294
Bottom Nusselt number: 7.0535586228536165
Top Nusselt number: 7.6511121496079335
Heat-flux mismatch (%): 7.8100217990522784
Temperature range: 0.0 to 1.0

Rayleigh number: 5000.0
Maximum speed: 1064.8532313417584
Mean speed: 296.3399545762971
RMS interior speed: 311.0596463012477
Bottom Nusselt number: 13.9276914008